In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score,mean_squared_error

In [ ]:
df = pd.read_csv("StudentPerformanceFactors.csv")

In [ ]:
print(df.shape)
print(df.sample(5))
print(df.isnull().sum())
print(df.describe())

In [ ]:
df.head()

numerical cols: hours_studied, attendance,sleep_hours, previous_scores,tutoring_sessions, , physical_activity,exam_scores
(7)
categorical_cols:parental_involvement,Accesstoresources,extracurricular_activitites,motivationlevel,familyincome,teacherquality,schooltype,peerinfluence,learningdisabilities,parentaleducation,distancefromhome,gender(13)

In [ ]:
print(df[df["Exam_Score"]>100])

In [ ]:
sns.histplot(df['Exam_Score'],bins =20)
plt.title("exam_score distribution")
plt.show()

In [ ]:
df = df[df['Exam_Score']<=100]
df.shape

In [ ]:
num_cols = ['Hours_Studied', 'Attendance', 'Sleep_Hours', 
            'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']
plt.figure(figsize=(12,8))
df[num_cols].hist(bins=20)
plt.title("numerical_columns_distribution")
plt.show()

In [ ]:
corr = df[num_cols + ['Exam_Score']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
cat_cols = ['Gender', 'School_Type', 'Motivation_Level', 
            'Family_Income', 'Parental_Involvement']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.boxplot(x=col, y='Exam_Score', data=df, ax=axes[i])
    axes[i].set_title(f'{col} vs Exam Score')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['Gender', 'School_Type', 'Motivation_Level', 
            'Family_Income', 'Parental_Involvement',
            'Access_to_Resources', 'Peer_Influence',
            'Learning_Disabilities', 'Internet_Access',
            'Extracurricular_Activities', 'Teacher_Quality',
            'Parental_Education_Level', 'Distance_from_Home']


fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.boxplot(x=col, y='Exam_Score', data=df, ax=axes[i])
    axes[i].set_title(f'{col} vs Exam Score')
    axes[i].tick_params(axis='x', rotation=45)


for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
num_cols = ['Hours_Studied', 'Attendance', 'Sleep_Hours', 
            'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], bins=20, kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)

plt.suptitle("Numeric Columns Distribution", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['Gender', 'School_Type', 'Motivation_Level', 
            'Family_Income', 'Parental_Involvement',
            'Access_to_Resources', 'Peer_Influence',
            'Learning_Disabilities', 'Internet_Access',
            'Extracurricular_Activities', 'Teacher_Quality',
            'Parental_Education_Level', 'Distance_from_Home']

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(x=col, data=df, ax=axes[i], palette='pastel')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].tick_params(axis='x', rotation=45)

# Hide empty subplots
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Columns Distribution", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(df['Exam_Score'], bins=20, kde=True, ax=axes[0], color='coral')
axes[0].set_title('Exam Score Distribution')

# Boxplot
sns.boxplot(y=df['Exam_Score'], ax=axes[1], color='coral')
axes[1].set_title('Exam Score Boxplot')

plt.suptitle("Target Variable Analysis", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(f'Outliers in {col}')
    axes[i].set_ylabel(col)

plt.suptitle("Outlier Detection - Numeric Columns", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# for col in num_cols:
#     Q1 = df[col].quantile(0.25)
#     Q3 = df[col].quantile(0.75)
#     IQR = Q3 - Q1
#     lower = Q1 - 1.5 * IQR
#     upper = Q3 + 1.5 * IQR
#     outliers = df[(df[col] < lower) | (df[col] > upper)]
#     print(f"{col}: {len(outliers)} outliers | Lower: {lower:.1f} | Upper: {upper:.1f}")

In [ ]:
df['Hours_Studied'] = df['Hours_Studied'].clip(upper=36)


In [ ]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [ ]:
num_cols = ['Hours_Studied', 'Attendance', 'Sleep_Hours',
            'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

ordinal_cols = ['Parental_Involvement', 'Access_to_Resources',
                'Motivation_Level', 'Family_Income', 'Teacher_Quality',
                'Parental_Education_Level', 'Distance_from_Home',
                'Peer_Influence']

nominal_cols = ['Gender', 'School_Type', 'Extracurricular_Activities',
                'Internet_Access', 'Learning_Disabilities']

In [ ]:
ordinal_orders = [
    ['Low', 'Medium', 'High'],                      
    ['Low', 'Medium', 'High'],                      
    ['Low', 'Medium', 'High'],                       
    ['Low', 'Medium', 'High'],                       
    ['Low', 'Medium', 'High'],                       
    ['High School', 'College', 'Postgraduate'],      
    ['Near', 'Moderate', 'Far'],                     
    ['Negative', 'Neutral', 'Positive'],             
]

In [ ]:
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=ordinal_orders))
])
nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


In [ ]:
preprocessor = ColumnTransformer([
    ('num',     num_pipeline,     num_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols),
    ('nominal', nominal_pipeline, nominal_cols)
])
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,random_state=42
)

In [ ]:
pipeline.fit(X_train,y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
r2_score(y_test,y_pred)

In [ ]:
np.sqrt(mean_squared_error(y_test,y_pred))

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    pipeline,X,y,cv=20,scoring='r2'     
)

print("R² for each fold:", cv_scores.round(4))
print(f"Mean R²         : {cv_scores.mean():.4f}")
print(f"Std R²          : {cv_scores.std():.4f}")

In [ ]:

cols_to_drop = ['Gender', 'School_Type', 'Internet_Access', 
                'Sleep_Hours', 'Physical_Activity', 'Distance_from_Home']

df_reduced = df.drop(columns=cols_to_drop)

print("Original shape:", df.shape)
print("Reduced shape :", df_reduced.shape)
print("\nRemaining columns:", df_reduced.columns.tolist())

In [ ]:

X_reduced = df_reduced.iloc[:, :-1]
y_reduced  = df_reduced.iloc[:, -1]

num_cols_r = ['Hours_Studied', 'Attendance', 
              'Previous_Scores', 'Tutoring_Sessions']

ordinal_cols_r = ['Parental_Involvement', 'Access_to_Resources',
                  'Motivation_Level', 'Family_Income', 'Teacher_Quality',
                  'Parental_Education_Level', 'Peer_Influence']

nominal_cols_r = ['Extracurricular_Activities', 'Learning_Disabilities']

In [ ]:

ordinal_orders_r = [
    ['Low', 'Medium', 'High'],                   
    ['Low', 'Medium', 'High'],                   
    ['Low', 'Medium', 'High'],                  
    ['Low', 'Medium', 'High'],                   
    ['Low', 'Medium', 'High'],                 
    ['High School', 'College', 'Postgraduate'],  
    ['Negative', 'Neutral', 'Positive'],         
]


num_pipeline_r = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

ordinal_pipeline_r = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=ordinal_orders_r))
])

nominal_pipeline_r = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_r = ColumnTransformer([
    ('num',     num_pipeline_r,     num_cols_r),
    ('ordinal', ordinal_pipeline_r, ordinal_cols_r),
    ('nominal', nominal_pipeline_r, nominal_cols_r)
])

# Full pipeline
full_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('model', Ridge())
])

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np
cv_original = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
cv_reduced = cross_val_score(full_pipeline_r, X_reduced, y_reduced, cv=5, scoring='r2')

print("Original Model (all features):")
print(f"  Scores : {cv_original.round(4)}")
print(f"  Mean R²: {cv_original.mean():.4f}")
print(f"  Std R² : {cv_original.std():.4f}")

print("\nReduced Model (dropped weak features):")
print(f"  Scores : {cv_reduced.round(4)}")
print(f"  Mean R²: {cv_reduced.mean():.4f}")
print(f"  Std R² : {cv_reduced.std():.4f}")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

models = {
    'Linear Regression'   : LinearRegression(),
    'Ridge Regression'    : Ridge(),
    'Lasso Regression'    : Lasso(),
    'Decision Tree'       : DecisionTreeRegressor(random_state=42),
    'Random Forest'       : RandomForestRegressor(random_state=42),
    'Gradient Boosting'   : GradientBoostingRegressor(random_state=42),
    'KNN'                 : KNeighborsRegressor()
}

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd

results = []

for name, model in models.items():
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
  
    r2_scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
    
    rmse_scores = -cross_val_score(pipeline, X, y, cv=5, 
                                   scoring='neg_root_mean_squared_error')
    
    results.append({
        'Model'    : name,
        'Mean R²'  : round(r2_scores.mean(), 4),
        'Std R²'   : round(r2_scores.std(), 4),
        'Mean RMSE': round(rmse_scores.mean(), 4),
        'Std RMSE' : round(rmse_scores.std(), 4)
    })
results_df = pd.DataFrame(results).sort_values('Mean R²', ascending=False)
print("\n", results_df.to_string(index=False))

In [ ]:
from sklearn.model_selection import GridSearchCV
ridge_params = {
    'model__alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 
                     10.0, 50.0, 100.0, 500.0, 1000.0]
}

ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_params,
    cv=5,
    scoring='r2',
    verbose=1
)

ridge_grid.fit(X, y)

print(f"\nBest Alpha    : {ridge_grid.best_params_}")
print(f"Best R² Score : {ridge_grid.best_score_:.4f}")
from sklearn.model_selection import RandomizedSearchCV

gb_params = {
    'model__n_estimators'  : [100, 200, 300],
    'model__learning_rate' : [0.01, 0.05, 0.1, 0.2],
    'model__max_depth'     : [2, 3, 4, 5],
    'model__min_samples_split': [2, 5, 10],
    'model__subsample'     : [0.8, 0.9, 1.0]
}

gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

gb_random = RandomizedSearchCV(
    gb_pipeline,
    gb_params,
    n_iter=20,        
    cv=5,
    scoring='r2',
    random_state=42,
    verbose=1
)

gb_random.fit(X, y)

print(f"\nBest Params   : {gb_random.best_params_}")
print(f"Best R² Score : {gb_random.best_score_:.4f}")
rf_params = {
    'model__n_estimators'  : [100, 200, 300],
    'model__max_depth'     : [None, 5, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__max_features'  : ['sqrt', 'log2', None]
}

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_random = RandomizedSearchCV(
    rf_pipeline,
    rf_params,
    n_iter=20,
    cv=5,
    scoring='r2',
    random_state=42,
    verbose=1
)

rf_random.fit(X, y)

print(f"\nBest Params   : {rf_random.best_params_}")
print(f"Best R² Score : {rf_random.best_score_:.4f}")